## Import Liberaries

In [22]:
import numpy as np
import rasterio
import tensorflow
from tensorflow import keras
from keras.layers import Input, Conv2D
# from keras import Sequential
# from keras.layers import Input, Flatten, Conv2D, Dense, MaxPool2D
from keras.models import Model

In [25]:
import os
import tensorflow as tf
import tifffile as tiff
from glob import glob

## Model Architecture

In [15]:
def build_srcnn(input_shape=(None, None, 3)):
    """
    Builds a 3-layer SRCNN model.

    Parameters:
        input_shape (tuple): Shape of input image (H, W, C)

    Returns:
        model (tf.keras.Model): SRCNN model
    """

    inputs = Input(shape=input_shape)

    # Layer 1: Patch extraction
    x = Conv2D(
        filters=64,
        kernel_size=(9, 9),
        strides=(1, 1),
        padding='same',
        activation='relu',
        name='conv1'
    )(inputs)

    # Layer 2: Non-linear mapping
    x = Conv2D(
        filters=32,
        kernel_size=(5, 5),
        strides=(1, 1),
        padding='same',
        activation='relu',
        name='conv2'
    )(x)

    # Layer 3: Reconstruction
    outputs = Conv2D(
        filters=3,
        kernel_size=(5, 5),
        strides=(1, 1),
        padding='same',
        activation='linear',
        name='conv3'
    )(x)

    model = Model(inputs=inputs, outputs=outputs, name='SRCNN')

    return model

In [24]:
# model2=Sequential([
#     Input(shape=(None, None, 3)),
#     Conv2D(64, (9, 9), padding='same', activation='relu'), # default value of stride=(1,1)
#     Conv2D(32, (5, 5), padding='same', activation='relu'),
#     Conv2D(3, (5, 5), padding='same', activation='linear')
# ])
# model2.summary()

In [20]:
# Create the model
model = build_srcnn(input_shape=(None, None,3))

# Display model summary
model.summary()

Model: "SRCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 512, 512, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 512, 512, 64)   │        15,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 512, 512, 32)   │        51,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 512, 512, 3)    │         2,403 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 69,251 (270.51 KB)

 Trainable params: 69,251 (270.51 KB)

 Non-trainable params: 0 (0.00 B)

## Creating data pipeline

In [10]:
# import tifffile as tiff
sample = tiff.imread(train_hr[0])

print(sample.dtype)
print(sample.min(), sample.max())

uint8
0 240


In [46]:
TRAIN_HR = "/kaggle/input/datasets/dozeradi007/patch-dataset-for-paired-hr-lr-satellite-imagery/patch_dataset/train/HR_0.5m"
TRAIN_LR = "/kaggle/input/datasets/dozeradi007/patch-dataset-for-paired-hr-lr-satellite-imagery/patch_dataset/train/LR_1m"

VAL_HR = "/kaggle/input/datasets/dozeradi007/patch-dataset-for-paired-hr-lr-satellite-imagery/patch_dataset/val/HR_0.5m"
VAL_LR = "/kaggle/input/datasets/dozeradi007/patch-dataset-for-paired-hr-lr-satellite-imagery/patch_dataset/val/LR_1m"

TEST_HR = "/kaggle/input/datasets/dozeradi007/patch-dataset-for-paired-hr-lr-satellite-imagery/patch_dataset/val/HR_0.5m"
TEST_LR = "/kaggle/input/datasets/dozeradi007/patch-dataset-for-paired-hr-lr-satellite-imagery/patch_dataset/val/LR_1m"

- now every 6th patch is irrespective to any resolution is smaller in size
  > format (H, W, C) I don't know why this happen
    - for HR ideal: (512, 512, 3), (n+6)th patch: (512, 440, 3)
    - for LR_1m ideal: (256, 256, 3), (n+6)th patch: (256, 220, 3)
    - for LR_2m ideal: (128, 128, 3), (n+6)th patch: (128, 110, 3)

- I have 2 solutions:
    - Filter the bad/edge patches, in get_image_pairs()`
        - SRCNN is fully convolutional model, so it can process images of any size at inference
    - Add padding to the edge patches, inside `preprocess()`
```
lr = tf.image.resize_with_crop_or_pad(lr, 256, 256)
hr = tf.image.resize_with_crop_or_pad(hr, 512, 512)
```
- for now I choose to filter the edge patches

In [34]:
def get_image_pairs(hr_root, lr_root):

    hr_paths = sorted(glob(os.path.join(hr_root, "*", "*.tif")))
    lr_paths = sorted(glob(os.path.join(lr_root, "*", "*.tif")))
    # assert len(hr_paths) == len(lr_paths), "HR and LR image count mismatch!"
    # return hr_paths, lr_paths
    
    good_hr = []
    good_lr = []

    for hr, lr in zip(hr_paths, lr_paths):

        hr_img = tiff.imread(hr)
        lr_img = tiff.imread(lr)

        if hr_img.shape == (512,512,3) and lr_img.shape == (256,256,3):
            good_hr.append(hr)
            good_lr.append(lr)

    return good_hr, good_lr

In [35]:
train_hr, train_lr = get_image_pairs(TRAIN_HR, TRAIN_LR)
val_hr, val_lr = get_image_pairs(VAL_HR, VAL_LR)

print("Training patches :", len(train_hr))
print("Validation patches :", len(val_hr))

Training patches : 4375
Validation patches : 925


In [36]:
def load_image(lr_path, hr_path):

    lr = tiff.imread(lr_path.decode())

    hr = tiff.imread(hr_path.decode())

    return lr.astype(np.float32), hr.astype(np.float32)

In [37]:
def preprocess(lr_path, hr_path):
# uses tf.numpy_function to wrap a regular Python/NumPy function (load_image) 
# so it can be used inside a TensorFlow data pipeline
    lr, hr = tf.numpy_function(
        load_image,
        [lr_path, hr_path],
        [tf.float32, tf.float32]
    )

    lr.set_shape([256,256,3])
    hr.set_shape([512,512,3])

    # Bicubic Upsampling
    lr = tf.image.resize(
        lr,
        [512,512],
        method='bicubic'
    )

    # Normalize

    lr = lr / 255.0
    hr = hr / 255.0

    return lr, hr

## Defining dataset

In [38]:
# BATCH_SIZE = 16
BATCH_SIZE = 8
# BATCH_SIZE = 4
# BATCH_SIZE = 2

AUTOTUNE = tf.data.AUTOTUNE


train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_lr, train_hr)
)

train_dataset = (
    train_dataset
    .shuffle(1000)
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


val_dataset = tf.data.Dataset.from_tensor_slices(
    (val_lr, val_hr)
)

val_dataset = (
    val_dataset
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [64]:
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(
#         learning_rate=1e-4
#     ),
#     loss='mse',
#     metrics=['mae']
# )

# Uses both the GPUs
strategy = tf.distribute.MirroredStrategy()

print("Number of devices:", strategy.num_replicas_in_sync)

with strategy.scope():
    model = build_srcnn()      # your model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss='mse',
        metrics=['mae']
    )

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of devices: 2


- Here's all the internal working
    - TensorFlow detects both GPUs
    - A **copy of the model** is created on each GPU. Initially, both copies have identical weights.
    - The **batch is split** automatically
        - If batch=32, it splits into 16, 16. Each GPU performs the forward and backward pass on its own portion of the batch.
    - After computing gradients: TensorFlow uses an efficient communication method (such as NCCL on NVIDIA GPUs) to **average the gradients**, applies a single optimizer update, and then keeps the model weights synchronized across both GPUs.

In [40]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_srcnn.keras",
    monitor="val_loss",
    save_best_only=True
)

earlystop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    # patience=10,
    patience=6,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    # patience=5
    patience=3
)

In [42]:
# for lr_path in train_lr[:20]:
#     img = tiff.imread(lr_path)
#     print(img.shape)

## Model training

In [43]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=100,
    callbacks=[checkpoint,
               earlystop,
               reduce_lr]
)

Epoch 1/100


2026-08-07 09:01:51.608425: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv (f32[8,32,512,512]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,64,512,512]{3,2,1,0}, f32[32,64,5,5]{3,2,1,0}, f32[32]{0}), window={size=5x5 pad=2_2x2_2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2026-08-07 09:01:52.508213: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.899905307s
Trying algorithm eng0{} for conv (f32[8,32,512,512]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,64,512,512]{3,2,1,0}, f32[32,64,5,5]{3,2,1,0}, f32[32]{0}), window={size=5x5 pad=2_2x2_2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationFor

546/547 ━━━━━━━━━━━━━━━━━━━━ 0s 356ms/step - loss: 0.0105 - mae: 0.0592

2026-08-07 09:05:43.173056: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv (f32[7,32,512,512]{3,2,1,0}, u8[0]{0}) custom-call(f32[7,64,512,512]{3,2,1,0}, f32[32,64,5,5]{3,2,1,0}, f32[32]{0}), window={size=5x5 pad=2_2x2_2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2026-08-07 09:05:43.813899: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.640959253s
Trying algorithm eng0{} for conv (f32[7,32,512,512]{3,2,1,0}, u8[0]{0}) custom-call(f32[7,64,512,512]{3,2,1,0}, f32[32,64,5,5]{3,2,1,0}, f32[32]{0}), window={size=5x5 pad=2_2x2_2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationFor

547/547 ━━━━━━━━━━━━━━━━━━━━ 0s 408ms/step - loss: 0.0104 - mae: 0.0592

2026-08-07 09:06:12.346835: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv (f32[8,32,512,512]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,64,512,512]{3,2,1,0}, f32[32,64,5,5]{3,2,1,0}, f32[32]{0}), window={size=5x5 pad=2_2x2_2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kRelu","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2026-08-07 09:06:13.204782: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.858082359s
Trying algorithm eng0{} for conv (f32[8,32,512,512]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,64,512,512]{3,2,1,0}, f32[32,64,5,5]{3,2,1,0}, f32[32]{0}), window={size=5x5 pad=2_2x2_2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationFor

547/547 ━━━━━━━━━━━━━━━━━━━━ 288s 454ms/step - loss: 0.0038 - mae: 0.0358 - val_loss: 0.0015 - val_mae: 0.0241 - learning_rate: 1.0000e-04
Epoch 2/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 208s 380ms/step - loss: 0.0014 - mae: 0.0241 - val_loss: 0.0013 - val_mae: 0.0223 - learning_rate: 1.0000e-04
Epoch 3/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 208s 381ms/step - loss: 0.0012 - mae: 0.0219 - val_loss: 0.0012 - val_mae: 0.0215 - learning_rate: 1.0000e-04
Epoch 4/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 209s 381ms/step - loss: 0.0012 - mae: 0.0215 - val_loss: 0.0015 - val_mae: 0.0278 - learning_rate: 1.0000e-04
Epoch 5/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 209s 382ms/step - loss: 0.0011 - mae: 0.0210 - val_loss: 0.0011 - val_mae: 0.0207 - learning_rate: 1.0000e-04
Epoch 6/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 208s 381ms/step - loss: 0.0011 - mae: 0.0209 - val_loss: 0.0011 - val_mae: 0.0205 - learning_rate: 1.0000e-04
Epoch 7/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 210s 383ms/step - loss: 0.0011 - mae: 0.0206 - val_loss: 0.0011 -

KeyboardInterrupt: 

In [ ]:
model.save("SRCNN_Final.keras")

## After stopping training

In [47]:
test_hr, test_lr = get_image_pairs(TEST_HR, TEST_LR)

In [45]:
stopped_model=keras.models.load_model("/kaggle/working/best_srcnn.keras")

In [49]:
test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_lr, test_hr)
)

test_dataset = (
    test_dataset
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [59]:
lr, hr = next(iter(test_dataset))

# Bicubic baseline
bicubic = tf.image.resize(lr, [512, 512], method="bicubic")

# SRCNN prediction
sr = model.predict(bicubic, verbose=0)

# PSNR
bicubic_psnr = tf.image.psnr(hr, bicubic, max_val=1.0)
srcnn_psnr = tf.image.psnr(hr, sr, max_val=1.0)

print("Bicubic PSNR:", bicubic_psnr.numpy().mean())
print("SRCNN PSNR :", srcnn_psnr.numpy().mean())

Bicubic PSNR: 24.800682
SRCNN PSNR : 26.081501
